# Value Iteration and Policy Iteration

## Learning Objectives
- Implement **value iteration** and prove it converges to $V^*$ via the contraction mapping theorem
- Implement **policy iteration**: policy evaluation + policy improvement, repeat until stable
- Understand why policy iteration converges in **fewer iterations** despite each step being more expensive
- Compare both algorithms on convergence speed and computational cost
- Understand **synchronous vs asynchronous** updates in value iteration

## Problem Statement

Both algorithms find $V^*$ and $\pi^*$ for a finite MDP $(S, A, \{P_{sa}\}, \gamma, R)$.

---

**Value iteration** (Bellman backup as fixed-point iteration):
$$V^{(t+1)}(s) \leftarrow R(s) + \max_{a}\; \gamma \sum_{s'} P_{sa}(s')\, V^{(t)}(s')$$

Converges because the Bellman backup $T$ is a **$\gamma$-contraction** in $\ell^\infty$:
$$\|TV - TV'\|_\infty \leq \gamma \|V - V'\|_\infty$$

So $\|V^{(t)} - V^*\|_\infty \leq \gamma^t \|V^{(0)} - V^*\|_\infty \to 0$.

---

**Policy iteration**:
1. **Policy evaluation**: solve $V^{\pi_k} = (I - \gamma P_{\pi_k})^{-1}R$ exactly (or iteratively)
2. **Policy improvement**: $\pi_{k+1}(s) = \arg\max_a \sum_{s'} P_{sa}(s')\, V^{\pi_k}(s')$
3. Repeat until $\pi_{k+1} = \pi_k$

Terminates because there are finitely many policies ($|A|^{|S|}$) and each step strictly improves or the policy is already optimal.

## 1. Value Iteration: Convergence and the Contraction Property

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 4x4 grid world (same as ml_001_24_1)
grid_size  = 4
n_states   = grid_size ** 2
n_actions  = 4
goal_state = n_states - 1

def state_to_rc(s, g=grid_size): return s // g, s % g
def rc_to_state(r, c, g=grid_size): return r * g + c

def build_transitions(g=grid_size):
    n = g * g
    P = np.zeros((n, 4, n))
    for s in range(n):
        r, c = state_to_rc(s, g)
        for a, (dr, dc) in enumerate([(-1,0),(1,0),(0,-1),(0,1)]):
            nr, nc = max(0, min(g-1, r+dr)), max(0, min(g-1, c+dc))
            P[s, a, rc_to_state(nr, nc, g)] = 1.0
    return P

P = build_transitions()
R = np.full(n_states, -1.0)
R[goal_state] = 10.0
gamma = 0.9

def bellman_backup(V, P, R, gamma):
    # Q[s, a] = R[s] + gamma * sum_{s'} P[s,a,s'] V[s']
    Q = R[:, None] + gamma * (P * V[None, None, :]).sum(axis=2)
    return Q.max(axis=1)

# Track convergence of value iteration
V = np.zeros(n_states)
V_hist   = [V.copy()]
err_hist = []

# First compute V* via many iterations to get ground truth
V_star = np.zeros(n_states)
for _ in range(2000):
    V_star = bellman_backup(V_star, P, R, gamma)

# Now run VI and track error
V = np.zeros(n_states)
for t in range(50):
    V_new = bellman_backup(V, P, R, gamma)
    err_hist.append(np.max(np.abs(V_new - V_star)))
    V = V_new
    V_hist.append(V.copy())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Convergence error
ax = axes[0]
ax.semilogy(range(1, len(err_hist)+1), err_hist, 'b-o', lw=2.5, ms=5)
# Theoretical bound: gamma^t * ||V^0 - V*||_inf
bound = [gamma**t * np.max(np.abs(V_star)) for t in range(1, len(err_hist)+1)]
ax.semilogy(range(1, len(bound)+1), bound, 'r--', lw=2, label=f'Bound: $\\gamma^t \\|V^*\\|_\\infty$')
ax.set_xlabel('Iteration $t$')
ax.set_ylabel('$\\|V^{(t)} - V^*\\|_\\infty$')
ax.set_title(f'Value iteration convergence ($\\gamma={gamma}$)\nGeometric rate = $\\gamma={gamma}$')
ax.legend(fontsize=9)

# V values at selected states over iterations
ax = axes[1]
selected = [0, 4, 8, 12, 15]
for s in selected:
    vals = [V_hist[t][s] for t in range(len(V_hist))]
    ax.plot(vals, lw=2, label=f's={s} (goal={goal_state})' if s == goal_state else f's={s}')
ax.axhline(0, color='k', ls=':', lw=1)
ax.set_xlabel('Iteration')
ax.set_ylabel('$V^{(t)}(s)$')
ax.set_title('Value function for selected states\n(converges to $V^*$)')
ax.legend(fontsize=8.5)

# Contraction: ||TV - TV'|| <= gamma * ||V - V'||
ax = axes[2]
np.random.seed(42)
V_a = np.random.randn(n_states) * 5
V_b = np.random.randn(n_states) * 5
diffs_before, diffs_after = [], []
for _ in range(30):
    diff_before = np.max(np.abs(V_a - V_b))
    TV_a = bellman_backup(V_a, P, R, gamma)
    TV_b = bellman_backup(V_b, P, R, gamma)
    diff_after = np.max(np.abs(TV_a - TV_b))
    diffs_before.append(diff_before)
    diffs_after.append(diff_after)
    V_a, V_b = TV_a, TV_b

ax.plot(diffs_before, 'r-', lw=2, label='$\\|V_a - V_b\\|_\\infty$')
ax.plot(diffs_after,  'b-', lw=2, label='$\\|TV_a - TV_b\\|_\\infty$')
ax.set_xlabel('Step')
ax.set_ylabel('$\\ell^\\infty$ distance')
ax.set_title(f'Contraction: $\\|TV-TV\'\\|_\\infty \\leq {gamma}\\|V-V\'\\|_\\infty$')
ax.legend(fontsize=9)
ratios = [a/b for a, b in zip(diffs_after, diffs_before) if b > 1e-8]
ax.text(0.5, 0.95, f'Empirical ratio ≈ {np.mean(ratios):.3f}  (theory: {gamma})',
        transform=ax.transAxes, ha='center', va='top', fontsize=9,
        bbox=dict(boxstyle='round', fc='lightyellow'))

fig.suptitle('Value Iteration: Convergence Rate and Contraction Property',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Policy Iteration: Evaluation + Improvement

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def policy_eval_exact(P, R, policy, gamma):
    """Exact policy evaluation: V = (I - gamma P_pi)^{-1} R."""
    n = len(R)
    P_pi = np.array([P[s, policy[s], :] for s in range(n)])
    return np.linalg.solve(np.eye(n) - gamma * P_pi, R)

def policy_improve(P, V, gamma):
    """Greedy policy improvement."""
    Q = gamma * (P * V[None, None, :]).sum(axis=2)  # (n, |A|)
    return Q.argmax(axis=1)

def policy_iteration(P, R, gamma, max_iters=100):
    n = len(R)
    policy = np.zeros(n, dtype=int)   # start: always action 0
    V_hist = []
    policy_hist = [policy.copy()]

    for it in range(max_iters):
        V  = policy_eval_exact(P, R, policy, gamma)
        V_hist.append(V.copy())
        pi_new = policy_improve(P, V, gamma)
        if np.all(pi_new == policy):
            break
        policy = pi_new
        policy_hist.append(policy.copy())

    return policy, V, V_hist, policy_hist

pi_star_pi, V_pi, V_hist_pi, pol_hist = policy_iteration(P, R, gamma)

n_iter_pi = len(V_hist_pi)
print(f'Policy iteration converged in {n_iter_pi} iterations')

fig, axes = plt.subplots(1, n_iter_pi + 1, figsize=(4 * (n_iter_pi + 1), 4))
action_arrows = ['↑', '↓', '←', '→']

for it, (V_it, pol_it) in enumerate(zip(V_hist_pi, pol_hist)):
    ax = axes[it]
    im = ax.imshow(V_it.reshape(grid_size, grid_size), cmap='YlGn')
    for s in range(n_states):
        r, c = state_to_rc(s)
        ax.text(c, r, f'{action_arrows[pol_it[s]]}\n{V_it[s]:.1f}',
                ha='center', va='center', fontsize=7.5)
    ax.set_title(f'Iteration {it}\nπ_{it}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

# Final (optimal)
ax = axes[n_iter_pi]
im = ax.imshow(V_pi.reshape(grid_size, grid_size), cmap='YlGn')
for s in range(n_states):
    r, c = state_to_rc(s)
    ax.text(c, r, f'{action_arrows[pi_star_pi[s]]}\n{V_pi[s]:.1f}',
            ha='center', va='center', fontsize=7.5)
ax.set_title(f'Converged: π*', fontsize=9)
ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(f'Policy Iteration: {n_iter_pi} Steps to Optimality',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Convergence Comparison: VI vs PI

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

def value_iteration_full(P, R, gamma, tol=1e-8, max_iters=5000):
    V = np.zeros(len(R))
    errs = []
    for _ in range(max_iters):
        Q = R[:, None] + gamma * (P * V[None, None, :]).sum(axis=2)
        V_new = Q.max(axis=1)
        errs.append(np.max(np.abs(V_new - V)))
        V = V_new
        if errs[-1] < tol:
            break
    return V, Q.argmax(axis=1), errs

def policy_iteration_full(P, R, gamma, max_iters=1000, tol=1e-8):
    n = len(R)
    policy = np.zeros(n, dtype=int)
    errs   = []
    for _ in range(max_iters):
        V = policy_eval_exact(P, R, policy, gamma)
        Q = gamma * (P * V[None, None, :]).sum(axis=2)
        pi_new = Q.argmax(axis=1)
        change = np.max(np.abs(
            np.array([Q[s, pi_new[s]] for s in range(n)]) -
            np.array([Q[s, policy[s]] for s in range(n)])
        ))
        errs.append(change)
        if np.all(pi_new == policy):
            break
        policy = pi_new
    V_final = policy_eval_exact(P, R, policy, gamma)
    return V_final, policy, errs

# Run on several gamma values
gammas = [0.5, 0.7, 0.9, 0.95, 0.99]
vi_iters, pi_iters = [], []

for g in gammas:
    _, _, errs_vi = value_iteration_full(P, R, g)
    _, _, errs_pi = policy_iteration_full(P, R, g)
    vi_iters.append(len(errs_vi))
    pi_iters.append(len(errs_pi))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Iterations vs gamma
ax = axes[0]
ax.plot(gammas, vi_iters, 'b-o', lw=2.5, ms=8, label='Value Iteration')
ax.plot(gammas, pi_iters, 'r-s', lw=2.5, ms=8, label='Policy Iteration')
ax.set_xlabel('Discount factor $\\gamma$')
ax.set_ylabel('Iterations to convergence')
ax.set_title('Convergence speed vs $\\gamma$\nPI needs far fewer iterations')
ax.legend(fontsize=9)

# Convergence error per iteration (gamma=0.9)
ax = axes[1]
_, _, errs_vi = value_iteration_full(P, R, 0.9)
_, _, errs_pi = policy_iteration_full(P, R, 0.9)
ax.semilogy(range(1, len(errs_vi)+1), errs_vi, 'b-', lw=2, label='Value Iteration')
ax.semilogy(range(1, len(errs_pi)+1), errs_pi, 'r-', lw=2, label='Policy Iteration')
ax.set_xlabel('Iteration')
ax.set_ylabel('Convergence measure (log scale)')
ax.set_title('Convergence error per iteration ($\\gamma=0.9$)')
ax.legend(fontsize=9)

# Comparison table
ax = axes[2]
ax.axis('off')
table_data = [
    ['Property', 'Value Iteration', 'Policy Iteration'],
    ['Each iteration', 'O(|S|²|A|)', 'O(|S|³ + |S|²|A|)'],
    ['Iterations needed', 'Many (∝ 1/(1-γ))', 'Few (finite policies)'],
    ['Per-step cost', 'Low', 'High (linear solve)'],
    ['Best when', '|S| huge, γ < 1', '|S| moderate, γ→1'],
    ['Convergence', 'Geometric (γ^t)', 'Quadratic near opt.'],
    ['Policy stable?', 'No (interleaved)', 'Yes (eval+improve)'],
]
table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                  cellLoc='center', loc='center',
                  colWidths=[0.35, 0.32, 0.33])
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 1.8)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2166ac')
        cell.set_text_props(color='white', fontweight='bold')
ax.set_title('Value Iteration vs Policy Iteration', fontsize=10, pad=80)

fig.suptitle('Convergence Comparison: Value Iteration vs Policy Iteration',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Iterations to convergence (tol=1e-8):')
print(f'{"γ":>6}  {"VI iters":>10}  {"PI iters":>10}')
for g, vi, pi in zip(gammas, vi_iters, pi_iters):
    print(f'{g:>6.2f}  {vi:>10d}  {pi:>10d}')

## 4. Synchronous vs Asynchronous Value Iteration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def vi_synchronous(P, R, gamma, max_iters=500, tol=1e-8):
    """Update all states simultaneously using V^{(t)} for all Q values."""
    V = np.zeros(len(R))
    errs = []
    for _ in range(max_iters):
        Q = R[:, None] + gamma * (P * V[None, None, :]).sum(axis=2)
        V_new = Q.max(axis=1)
        errs.append(np.max(np.abs(V_new - V)))
        V = V_new
        if errs[-1] < tol: break
    return V, errs

def vi_asynchronous(P, R, gamma, order, max_sweeps=200, tol=1e-8):
    """Update states one at a time in given order; use latest V immediately."""
    V = np.zeros(len(R))
    errs = []
    for _ in range(max_sweeps):
        V_old = V.copy()
        for s in order:
            q_s = R[s] + gamma * (P[s, :, :] * V[None, :]).sum(axis=1)
            V[s] = q_s.max()
        errs.append(np.max(np.abs(V - V_old)))
        if errs[-1] < tol: break
    return V, errs

# Compute ground-truth V* for error measurement
V_opt, _ = vi_synchronous(P, R, gamma, max_iters=5000, tol=1e-12)
V_opt = V_opt  # reference

# Different orderings for async VI
np.random.seed(0)
order_fwd  = list(range(n_states))
order_rev  = list(range(n_states - 1, -1, -1))  # goal first
order_rand = list(np.random.permutation(n_states))

V_sync, errs_sync = vi_synchronous(P, R, gamma)
_, errs_async_fwd  = vi_asynchronous(P, R, gamma, order_fwd)
_, errs_async_rev  = vi_asynchronous(P, R, gamma, order_rev)
_, errs_async_rand = vi_asynchronous(P, R, gamma, order_rand)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.semilogy(errs_sync,       'b-',  lw=2.5, label=f'Sync (sweeps={len(errs_sync)})')
ax.semilogy(errs_async_fwd,  'r-',  lw=2,   label=f'Async forward (sweeps={len(errs_async_fwd)})')
ax.semilogy(errs_async_rev,  'g-',  lw=2,   label=f'Async goal-first (sweeps={len(errs_async_rev)})')
ax.semilogy(errs_async_rand, 'k--', lw=2,   label=f'Async random (sweeps={len(errs_async_rand)})')
ax.set_xlabel('Sweep (one pass over all states)')
ax.set_ylabel('Max change (log scale)')
ax.set_title(f'Sync vs Async VI ($\\gamma={gamma}$)\nAsync uses latest values immediately')
ax.legend(fontsize=8.5)

ax = axes[1]
# Visualise value propagation: show when each state first reaches near-optimal
V_track = np.zeros(n_states)
iters_to_converge = np.full(n_states, np.nan)
V_star_ref = V_opt

for t in range(200):
    Q = R[:, None] + gamma * (P * V_track[None, None, :]).sum(axis=2)
    V_track = Q.max(axis=1)
    for s in range(n_states):
        if np.isnan(iters_to_converge[s]) and abs(V_track[s] - V_star_ref[s]) < 1.0:
            iters_to_converge[s] = t + 1

grid = iters_to_converge.reshape(grid_size, grid_size)
im = ax.imshow(grid, cmap='RdYlGn_r')
plt.colorbar(im, ax=ax, label='Iteration first converged')
for s in range(n_states):
    r, c = state_to_rc(s)
    ax.text(c, r, f'{int(iters_to_converge[s]) if not np.isnan(iters_to_converge[s]) else "?"}',
            ha='center', va='center', fontsize=9)
ax.set_title('Iterations for each state to first reach\nnear-optimal value (sync VI)')
ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Synchronous vs Asynchronous Value Iteration',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Bellman</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">optimality eq.</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >V*(s) = max&#x2090; [R(s) + &#x3b3; &#x2211;&#x209b;&#x2019; P&#x209b;&#x2090;(s&#x2019;) V*(s&#x2019;)] &#x2014; fixed point</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >two algorithms to find V*</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Value</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">iteration</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >V&#x207b;&#x1d57;&#x207a;&#xb9;&#x207e;(s) := max&#x2090; [R(s) + &#x3b3; &#x2211;&#x209b;&#x2019; P&#x209b;&#x2090;(s&#x2019;) V&#x207b;&#x1d57;&#x207e;(s&#x2019;)] &#x2014; Bellman backup = contraction</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >converges in O(1/(1-&#x3b3;)) iters</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Policy</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">iteration</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >alternate: evaluate V&#x3c0; exactly, then improve &#x3c0; := greedy(V&#x3c0;)</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >fewer iterations than VI, each costlier</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Converged V*</text>
  <text x="102" y="352" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">&#x3c0;*</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >V* unique (contraction); &#x3c0;*(s) = argmax&#x2090; &#x2211;&#x209b;&#x2019; P&#x209b;&#x2090;(s&#x2019;) V*(s&#x2019;)</text>
</svg>
""")

## Summary

| Algorithm | Update rule | Convergence |
|---|---|---|
| Value iteration | $V^{(t+1)}(s) := \max_a\left[R(s) + \gamma\sum_{s'} P_{sa}(s')V^{(t)}(s')\right]$ | $\|V^{(t)} - V^*\|_\infty \to 0$ (contraction, $\gamma < 1$) |
| Policy iteration | Evaluate $V^\pi$, then $\pi' := \text{greedy}(V^\pi)$; repeat | Converges in finite steps (bounded policy count) |
| Policy evaluation | Solve linear system $V^\pi = R + \gamma P^\pi V^\pi$ | Exact; $O(\lvert S\rvert^3)$ |
| Policy improvement | $\pi'(s) := \arg\max_a \sum_{s'} P_{sa}(s') V^\pi(s')$ | Guaranteed: $V^{\pi'} \geq V^\pi$ pointwise |

**Key insight:** the Bellman backup operator is a $\gamma$-contraction — each application brings $V$ strictly closer to $V^*$; value iteration applies it directly while policy iteration alternates between exact evaluation and greedy improvement, converging faster in practice.